# RepoVeritas — Frontier datapoint (Claude Opus)

No GPU. No model download. Just ~207 API calls to Anthropic (~3 min). Same prompt/harness as the
Qwen sweep, so the result is directly comparable.

**Steps:**
1. Runtime can stay on CPU (no GPU needed).
2. Add your Anthropic key to **Secrets** (the 🔑 icon, left sidebar): name it `ANTHROPIC_API_KEY`,
   paste the key as the value, toggle **Notebook access** on. (This keeps the key out of the cells.)
3. Run the install cell.
4. Run the upload cell → upload **`baseline_eval.py`**, **`test_public.jsonl`**, **`test_gold.jsonl`**.
5. Run the eval cell. Download `frontier_opus.json` + `frontier_opus.txt` when done.

In [ ]:
!pip install -q anthropic

## Upload the 3 files
`baseline_eval.py`, `test_public.jsonl`, `test_gold.jsonl` (the ones you already have).

In [ ]:
from google.colab import files
print("Upload baseline_eval.py, test_public.jsonl, test_gold.jsonl ...")
files.upload()
# if baseline_eval uploads as "baseline_eval (1).py", fix the name:
import glob, os
for f in glob.glob("baseline_eval*.py"):
    if f != "baseline_eval.py": os.rename(f, "baseline_eval.py")
print("files:", [f for f in os.listdir() if f.endswith(('.py','.jsonl'))])

## Run the frontier eval
Pulls the key from Secrets, runs all 207 test items, prints the full report (incl. confusion matrix),
and saves `frontier_opus.json` + `frontier_opus.txt`.

In [ ]:
import os, io, contextlib
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

from baseline_eval import load_public, load_gold, build_prompt, parse_label, report, make_model_fn

MODEL = "anthropic:claude-opus-4-8"      # or "anthropic:claude-sonnet-4-6" if cost matters
pub  = load_public("test_public.jsonl")
gold = load_gold("test_gold.jsonl")
fn   = make_model_fn(MODEL)

from tqdm.auto import tqdm
rows = []
for it in tqdm(pub, desc=MODEL):
    raw = fn(build_prompt(it))
    g = gold[it["id"]]
    rows.append((g["label"], parse_label(raw), g["task_family"]))

# capture the printed report to a file as well as showing it
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    res = report(MODEL, rows)
out = buf.getvalue()
print(out)
open("frontier_opus.txt","w").write(out)
import json
json.dump(res, open("frontier_opus.json","w"), indent=2)
print("\nsaved frontier_opus.json + frontier_opus.txt")

In [ ]:
from google.colab import files
files.download("frontier_opus.json")
files.download("frontier_opus.txt")